# IDX Improved LightGBM


In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pandas.tseries.offsets import DateOffset
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
import warnings
warnings.filterwarnings('ignore')

## Load and Merge Data

In [17]:
df1 = pd.read_csv('idx_cleaned_data1.csv')
df2 = pd.read_csv('idx_cleaned_data2.csv', low_memory=False)
df3 = pd.read_csv('idx_cleaned_data3.csv', low_memory=False)
df  = pd.merge(df1, df2, how='outer')
df  = pd.merge(df,  df3, how='outer')
print(f"Merged dataset: {df.shape[0]:,} rows, {df.shape[1]} columns")

Merged dataset: 123,028 rows, 51 columns


## Outlier Removal

In [18]:
Q1 = df['ClosePrice'].quantile(0.005)
Q3 = df['ClosePrice'].quantile(0.995)
df = df[(df['ClosePrice'] >= Q1) & (df['ClosePrice'] <= Q3)]
print(f"After outlier removal: {len(df):,} rows")
print(f"ClosePrice range: ${df['ClosePrice'].min():,.0f} – ${df['ClosePrice'].max():,.0f}")

After outlier removal: 121,810 rows
ClosePrice range: $1,500 – $6,800,000


## Train / Test Split

In [19]:
#Time based split

df['CloseDate'] = pd.to_datetime(df['CloseDate'], yearfirst=True)
last_date  = df['CloseDate'].max()
test_date  = last_date - DateOffset(months=1)
train_date = test_date - DateOffset(months=6)

train_df = df[(df['CloseDate'] > train_date) & (df['CloseDate'] < test_date)].copy()
test_df  = df[df['CloseDate'] >= test_date].copy()
test_df  = test_df[
    (test_df['ClosePrice'] > test_df['ClosePrice'].quantile(0.05)) &
    (test_df['ClosePrice'] < test_df['ClosePrice'].quantile(0.95))
].copy()

print(f"Train: {len(train_df):,} rows  |  Test: {len(test_df):,} rows")
print(f"Train window: {train_date.date()} - {test_date.date()}")
print(f"Test  window: {test_date.date()} - {last_date.date()}")

Train: 99,165 rows  |  Test: 20,375 rows
Train window: 2024-12-30 - 2025-06-30
Test  window: 2025-06-30 - 2025-07-31


## Label Encoders


In [20]:
# Fit on full dataset so that test set doesn't encounter an unknown category.
le_zip     = LabelEncoder().fit(df['PostalCode'].fillna('Unknown').astype(str))
le_subtype = LabelEncoder().fit(df['PropertySubType'].fillna('Unknown').astype(str))
le_county  = LabelEncoder().fit(df['CountyOrParish'].fillna('Unknown').astype(str))
print(f"Unique zip codes:       {len(le_zip.classes_)}")
print(f"Unique property subtypes: {len(le_subtype.classes_)}")
print(f"Unique counties:        {len(le_county.classes_)}")

Unique zip codes:       1924
Unique property subtypes: 34
Unique counties:        56


## Feature Engineering


In [37]:
def build_features(d, le_zip, le_subtype, le_county):
    d = d.copy()

    #  Boolean columns
    bool_cols = ['ViewYN', 'PoolPrivateYN', 'AttachedGarageYN',
                 'FireplaceYN', 'NewConstructionYN']
    for col in bool_cols:
        d[col] = (d[col].astype(str).str.lower() == 'true').astype(int)

    # Property
    d['is_sfr'] = (
        d['Group'].astype(str).str.lower() == 'single family residential'
    ).astype(int)

    # Numeric features
    d['bbratio'] = (
        d['BathroomsTotalInteger'] / d['BedroomsTotal'].replace(0, 1)
    ).fillna(0)

    d['age'] = 2025 - d['YearBuilt']

    d['total_rooms'] = (
        d['BedroomsTotal'].fillna(0) + d['BathroomsTotalInteger'].fillna(0)
    )

    d['lot_to_living'] = (
        d['LotSizeSquareFeet'] / d['LivingArea'].replace(0, np.nan)
    ).fillna(0).clip(0, 100)

    d['hoa_monthly'] = d['AssociationFee'].fillna(0)
    d.loc[d['AssociationFeeFrequency'] == 'Annually',     'hoa_monthly'] /= 12
    d.loc[d['AssociationFeeFrequency'] == 'Quarterly',    'hoa_monthly'] /= 3
    d.loc[d['AssociationFeeFrequency'] == 'SemiAnnually', 'hoa_monthly'] /= 6


    # Final features
    feature_cols = [
        # Property size
        'LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger',
        'BuildingAreaTotal', 'LotSizeSquareFeet',
        # Location
        'Latitude', 'Longitude',
        # Property details
        'YearBuilt', 'age', 'Stories', 'ParkingTotal', 'GarageSpaces',
        'MainLevelBedrooms',
        # Ratios
        'bbratio', 'total_rooms', 'lot_to_living',
        'hoa_monthly',
        # Amenities
        'ViewYN', 'PoolPrivateYN', 'AttachedGarageYN',
        'FireplaceYN', 'NewConstructionYN', 'is_sfr',

    ]
    return d[feature_cols].fillna(0)

X_train = build_features(train_df, le_zip, le_subtype, le_county)
X_test  = build_features(test_df,  le_zip, le_subtype, le_county)
y_train = train_df['ClosePrice']
y_test  = test_df['ClosePrice']
print(f"Feature matrix — Train: {X_train.shape}  |  Test: {X_test.shape}")
print("\nFeatures:", X_train.columns.tolist())

Feature matrix — Train: (99165, 23)  |  Test: (20375, 23)

Features: ['LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'BuildingAreaTotal', 'LotSizeSquareFeet', 'Latitude', 'Longitude', 'YearBuilt', 'age', 'Stories', 'ParkingTotal', 'GarageSpaces', 'MainLevelBedrooms', 'bbratio', 'total_rooms', 'lot_to_living', 'hoa_monthly', 'ViewYN', 'PoolPrivateYN', 'AttachedGarageYN', 'FireplaceYN', 'NewConstructionYN', 'is_sfr']


### Model Training


In [38]:
model = LGBMRegressor(
    n_estimators       = 2000,
    max_depth          = 5,
    num_leaves         = 20,
    learning_rate      = 0.05,
    subsample          = 0.5,
    subsample_freq     = 1,
    colsample_bytree   = 0.8,
    min_child_samples  = 20,
    min_split_gain     = 1.0,
    objective          = 'regression',
    metric             = 'rmse'
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[
        early_stopping(stopping_rounds=50, verbose=False),
        log_evaluation(period=100),
    ],
)
print(f"\nBest iteration: {model.best_iteration_}")

[100]	valid_0's rmse: 275948
[200]	valid_0's rmse: 264580
[300]	valid_0's rmse: 258710
[400]	valid_0's rmse: 255618
[500]	valid_0's rmse: 252984
[600]	valid_0's rmse: 250655
[700]	valid_0's rmse: 248875
[800]	valid_0's rmse: 247287
[900]	valid_0's rmse: 246125
[1000]	valid_0's rmse: 244904
[1100]	valid_0's rmse: 244105
[1200]	valid_0's rmse: 243206
[1300]	valid_0's rmse: 242834
[1400]	valid_0's rmse: 242153
[1500]	valid_0's rmse: 241386
[1600]	valid_0's rmse: 240687
[1700]	valid_0's rmse: 239995
[1800]	valid_0's rmse: 239461
[1900]	valid_0's rmse: 239412

Best iteration: 1864


In [39]:
y_pred = model.predict(X_test)

train_r2  = r2_score(y_train, model.predict(X_train))
test_r2   = r2_score(y_test,  y_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Train R2:  {train_r2:.4f}")
print(f"Test  R2:  {test_r2:.4f}")
print(f"Test  RMSE: ${test_rmse:,.2f}")

Train R2:  0.8958
Test  R2:  0.8215
Test  RMSE: $239,260.48


## Cross-Validation

In [35]:
cv_scores = cross_val_score(
    LGBMRegressor(
        n_estimators=500, max_depth=5, num_leaves=20,
        learning_rate=0.05, subsample=0.5, subsample_freq=1,
        colsample_bytree=0.8, min_child_samples=20,
        min_split_gain=1.0,
        objective='regression', random_state=42
    ),
    X_train, y_train,
    cv=3, scoring='r2'
)
print(f"3-Fold CV R2: {cv_scores.mean():.4f} ~ {cv_scores.std():.4f}")

3-Fold CV R2: 0.7646 ~ 0.0309
